# ComicAnalizer - Magi + PaddleOCR Pipeline

Este notebook genera la salida base para trabajar: detecciones Magi, reporte de calidad y comparacion OCR complementaria con PaddleOCR.

Antes de correrlo, activa GPU en Colab: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5

In [ ]:
%cd /content/ComicAnalizer
!pip -q install \
  "transformers==4.49.0" \
  "huggingface_hub<1.0" \
  timm \
  einops \
  pytorch-metric-learning \
  shapely \
  "paddleocr==3.3.3" \
  "paddlepaddle==3.2.0"

## Subir dataset limpio

Sube `magi_clean_full.zip`, generado localmente desde `outputs/magi_clean_full.zip`.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
print('ZIP subido:', zip_name)

!rm -rf /content/magi_sample
!mkdir -p /content/magi_sample
!unzip -q -o "$zip_name" -d /content/magi_sample
!find /content/magi_sample -maxdepth 4 -type d | head -30

## Ejecutar Magi

Para prueba corta cambia `MAX_COMICS = 2`. Para todo el dataset usa `MAX_COMICS = 0`.

In [ ]:
MAX_COMICS = 0

!python -m tools.inspect_magi_dataset \
  --input /content/magi_sample/magi_clean_full/by_comic \
  --output-dir outputs/magi_debug/colab_clean_full_detections \
  --all-pages-per-comic \
  --dataset-name test_1_clean \
  --task detections \
  --cache-dir outputs/magi_cache \
  --device cuda \
  --dtype float16 \
  --max-comics $MAX_COMICS

## Generar reporte normalizado de calidad

In [ ]:
!python -m tools.analyze_magi_results \
  --input outputs/magi_debug/colab_clean_full_detections \
  --output outputs/magi_analysis_report.json \
  --top-n 20

In [ ]:
import json
from pathlib import Path

report = json.loads(Path('outputs/magi_analysis_report.json').read_text())
print(json.dumps(report['summary'], indent=2, ensure_ascii=False))
print('\nPaginas sospechosas:', report['summary']['suspicious_page_count'])
print('\nTop flags:', report['summary']['flag_counts'])

## Comparar Magi contra PaddleOCR

Corre una muestra aleatoria primero. Sube `OCR_LIMIT` despues si el tiempo es razonable.

In [ ]:
OCR_LIMIT = 8

!python -m tools.compare_magi_paddleocr \
  --magi-input outputs/magi_debug/colab_clean_full_detections \
  --image-root /content/magi_sample/magi_clean_full/by_comic \
  --dataset-name test_1_clean \
  --selection random \
  --limit $OCR_LIMIT \
  --seed 42 \
  --lang en \
  --output outputs/paddle_magi_ocr_comparison.json

In [ ]:
ocr_report = json.loads(Path('outputs/paddle_magi_ocr_comparison.json').read_text())
print(json.dumps(ocr_report['summary'], indent=2, ensure_ascii=False))
for item in ocr_report['comparisons'][:10]:
    print(item['comic_id'], item['file_name'], 'Magi=', item['magi_text_regions'], 'Paddle=', item['paddle_text_blocks'], 'match=', item['matched_regions'], 't=', round(item['paddle_elapsed_seconds'], 2))

## Descargar salida estandar

In [ ]:
from google.colab import files

!zip -qr comic_analyzer_magi_ocr_outputs.zip \
  outputs/magi_debug/colab_clean_full_detections \
  outputs/magi_cache \
  outputs/magi_analysis_report.json \
  outputs/paddle_magi_ocr_comparison.json

files.download('comic_analyzer_magi_ocr_outputs.zip')